# A real example: identify moth species

<center><img src="images/moths.png" width=1000/></center>

## Transfer learning and fine-tuning
In the previous notebook we created our own very simple network and trained it from scratch. In practice, we wouldn't normally do this. Instead, we use an existing pre-trained model and adapt it two our needs using two techniques:
- Transfer learning: replace the classifier (or "head") with a new one that suits our domain
- Fine-tuning: further train the whole network on images from our domain

In [ ]:
# Import the libraries we need
import os
from glob import glob
import numpy as np
import tensorflow as tf # TODO try using tf2 to confirm compatibility?
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# import keras
from tensorflow import keras

# Keras models
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.mobilenet import MobileNet
from tensorflow.keras.applications.xception import Xception

# Keras utilities
import tensorflow.keras.backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

In [ ]:
from scripts import utils
# %load files/utils

## Re-using an existing model
The moths dataset is very small (around 250 images in total), which is far too small to learn the 11 classes and would result in severe overfitting if we tried. To get around this, we will preload a network with pre-trained weights from the ImageNet dataset. This is a huge image set that classifies photographs into thousands of classes. We will "transfer" its feature extraction (convolutional) layers to our moths domain by training a new "head" network (a densly connected neural network). We will then "fine-tune" the entire model to our domain.

Here's the code for setting up the network. We will use an Xception network; this is a high-performing mid-sized network.

In [ ]:
# Global constants for the Xception model
FREEZE_XCEPTION = (0,133)
IMAGE_SIZE_XCEPTION = (299,299)
IMAGE_SHAPE_XCEPTION = IMAGE_SIZE_XCEPTION + (3,)

def xception_model(num_classes):
    base_model = Xception(include_top=False, weights='imagenet', input_shape=IMAGE_SHAPE_XCEPTION)
    x = base_model.output
    x = GlobalAveragePooling2D()(x) # averages the values of each filter

    # Create the new dense top layer on top of the pre-trained feature extraction network
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.inputs, outputs=predictions)
    return model
    

## Transfer learning
We will now train the new "head" to use the output from pre-trained feature extraction to decide the class (moth species) of each image. To do this, we "freeze" all the feature extraction layers and then train the netowrk in the usual manner.

In [ ]:
def transfer(model, epochs, train_generator, val_generator, lr):
    # Freeze all layers except the new head
    for i in range(FREEZE_XCEPTION[0], FREEZE_XCEPTION[1]):
        model.layers[i].trainable = False 
        
    # compile model for training
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()
    
    # Now train the head
    history = model.fit_generator(train_generator, epochs=epochs, validation_data=val_generator)
    return history


## Fine-tuning
Once the new head has been trained to generate our domain's classes, the final step is to "fine-tune" the (pre-trained) feature extraction layers to match our imagery. We do this very gently; we wich to retain as much of the pre-trained richness in the model as possible, while tuning to to the pecurlarities of our particular imagery.

In [ ]:
def fine_tune(model, epochs, train_generator, val_generator, lr):
    for i in range(FREEZE_XCEPTION[0], FREEZE_XCEPTION[1]):
        model.layers[i].trainable = True
      
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='categorical_crossentropy', metrics=['accuracy'])     
    history = model.fit_generator(train_generator, epochs=epochs, validation_data=val_generator)
    return history


## Processing the images
In the previous notebook, we rescaled images to the range of 0..1 by dividing by 255. This time, we need to do things a little differently:
- We will be using a pretrained model that expects the input to be in the range -1..1
- We will normalise the imagery's brightness and contrast to +/- 1SD. This is common paractice for classification of photographs, as it avoids the model "learning" that different camera setups predict different classes.
- We will augment the imagery using some custom processing

Note that the keras libraries (and 3rd party extensions) contain much of this functionality - it's always worth looking for a pre-cooked version before writing your own code.

In [ ]:
def preprocess_input_MWLR(sample, augment=False):
  """
  Rescales the input data to match the expectation of the network.
  If 'augment' is true, also augment the image brightness and contrast (for training)
  """
  
  min = np.min(sample)
  max = np.max(sample)

  std = np.std(sample)
  if std == 0: std = 1.0 # avoid zero division
  mean = np.mean(sample)

  if augment:
    # Augment brightness (centre) and contrast (spread)
    STD_AUG = 0.2 # proportion to increase spread by
    MEAN_AUG = 0.2 # amount to displace centre by
    spread_shift = (random.random() - 0.5) * STD_AUG 
    centre_shift = mean * (random.random() - 0.5) * MEAN_AUG 
  else:
    # No augmentation
    spread_shift = 0
    centre_shift = 0
    
  x = ((sample - mean) * (1 + spread_shift) * 0.5/std) + centre_shift # 2SDs between 0 and 1
  
  return x


def preprocess_input_train_MWLR(sample):
  # For training we augment the images
  return preprocess_input_MWLR(sample, augment=True)

## Putting it all together
Now we're ready to put the pieces together, giving us our fine-tuner.

In [ ]:
def retrain(train_data_dir, valid_data_dir, model_save_dir, dataset_name, epochs=250, batch_size=16, lr=1e-4):

    '''
    Retrain an existing network by transfer learning and fine-tuning.
    '''
    
    # Create the model
    num_classes = len(os.listdir(train_data_dir))
    model = xception_model(num_classes)

    # Set up the data generators
    train_datagen = ImageDataGenerator(preprocessing_function = preprocess_input_train_MWLR, # standardise and augment brightness 
                                       width_shift_range = 0.8,
                                       height_shift_range = 0.8,
                                       zoom_range = 0.8,
                                       horizontal_flip = True)
    
    valid_datagen = ImageDataGenerator(preprocessing_function = preprocess_input_MWLR)
    
    train_generator = train_datagen.flow_from_directory(train_data_dir, target_size=IMAGE_SIZE_XCEPTION, batch_size = batch_size)
    valid_generator = valid_datagen.flow_from_directory(valid_data_dir, target_size=IMAGE_SIZE_XCEPTION, batch_size = batch_size)    

    # "Transfer" the model to the new domain
    print("\nTransfer training...")
    tr_history = transfer(model, epochs, train_generator, valid_generator, lr)

    # Fine-tune the entire model
    print("\nFine-tuning...")
    ft_history = fine_tune(model, epochs, train_generator, valid_generator, lr)

    model_file_name = os.path.join(model_save_dir, f"{dataset_name}_Xception_{epochs}_epoch_transfer.h5")
    model.save(model_file_name)

    # All done, clear keras' global state to avoid memory leaks
    K.clear_session()
    
    print(f"\nModel training complete. Model saved to {model_file_name}")
    return tr_history, ft_history

# Training the moths model
We will run the fine-tuning briefly to get an idea how it works. Note that to train the model properly would take considerably longer unless a GPU is used.

In [ ]:
DATA_PATH = 'files/moths_data/'
 
train_data_dir = DATA_PATH + 'Train'
validation_data_dir = DATA_PATH + 'Val'
labels_path = DATA_PATH + 'labels.txt'
model_dir = 'models/'
dataset_name = 'moths'

# epochs=10 # Just a quick test of the model training
epochs = 2  # CPU only
tr_hist, ft_hist = retrain(train_data_dir, validation_data_dir, model_dir, dataset_name, epochs=epochs, lr=1e-4)

In [ ]:
utils.plot_history(tr_hist, epochs, accuracy='accuracy')

In [ ]:
utils.plot_history(ft_hist, epochs, accuracy='accuracy')

Let's test our minimally trained model and see how it did.


In [ ]:
def run_model(model_path, labels_path, test_images_path, reset=False):
  """
  Runs the model on a folder of images (in subfolders by class). 
  """
  image_files = sorted(list(glob('{}/*/*.*'.format(test_images_path)))) 
  
  print("\nTesting model {} on {} ({} images):".format(model_path, test_images_path, len(image_files)))
    
  with open(labels_path, 'r') as f:
    labels = [l for l in f.read().split('\n')]

  print('Loading model "{}"'.format(model_path))
  model = keras.models.load_model(model_path)
  
  print('Predicting image classes...')
  datagen = ImageDataGenerator(preprocessing_function = preprocess_input_MWLR)
  data_generator = datagen.flow_from_directory(test_images_path, target_size=IMAGE_SIZE_XCEPTION, shuffle=False, batch_size=1)
  predictions = model.predict_generator(data_generator, steps = len(image_files))
  print('Prediction complete.')

  # All done, clear keras' global state to avoid memory leaks
  if reset:
    K.clear_session()

  return labels, predictions, image_files, model

In [ ]:
def test_model(model_path, labels_path, test_images_path):
  """
  Tests the model on a folder of images. 
  NOTE: Assumes the images are arranged into folders by class for evaluation purposes.
  """

  labels, predictions, image_files, model = run_model(model_path, labels_path, test_images_path)
  
  correct = 0
  wrong = 0

  answers = []
  
  print('\nResults:')
  print('---------------------------------------------------------------------------------------------------------')
  print("Classes: {}".format(labels))
  for file_num, p in enumerate(predictions):
    best = 0
    second = 0
    best_index = -1
    for i, pr in enumerate(p):
        if pr > best:
            best = pr
            best_index = i
    file_path = os.path.relpath(image_files[file_num], test_images_path)
    filename = os.path.basename(file_path)
    class_name = os.path.dirname(file_path)
    answers.append((class_name, labels[best_index], best, file_path, p)) # (actual, predicted)
    if labels[best_index] == class_name:
      outcome = 'CORRECT'
      correct += 1
    else:
      outcome = f'***WRONG*** predicted:"{labels[best_index]}" actual:"{class_name}"'
      wrong += 1
      
    print('\nPrediction for {} :{} => {}   {}'.format(file_path, p, labels[best_index], outcome))

  print("\nResults for model {} on {} ({} images):".format(model_path, test_images_path, len(image_files)))
  print('\nCORRECT: {0} ({1:.2f})%  WRONG: {2} ({3:.2f})%'.format(correct, 100* correct/(correct+wrong), wrong, 100* wrong/(correct+wrong))) 
  print('---------------------------------------------------------------------------------------------------------')
  
  # All done, clear keras' global state to avoid memory leaks
  K.clear_session()

  return labels, answers, model

In [ ]:
# Test the model on the test image set

test_data_dir = DATA_PATH + 'Test'
model_path = model_dir + 'moths_Xception_10_epoch_transfer.h5' # Our 10-epoch model
labels, answers, model = test_model(model_path, labels_path, test_data_dir)

In [ ]:
utils.confusion(answers)

We only trained the model for a short time. Let's see how the same model performs after it was trained for 100 epochs instead of just 10. This is still only about 15 minutes training on a GPU (NVIDIA GTX 1080Ti).

In [ ]:
# Test a model trained previously for 100 epochs on the test image set

test_data_dir = DATA_PATH + 'Test'
model_path = model_dir + 'moths_Xception_100_epoch_transfer.h5' # a 100-epoch model
labels, answers, model = test_model(model_path, labels_path, test_data_dir)

In [ ]:
utils.confusion(answers)

How has this model done?
- Where are the errors? Are these expected?
- How does the pattern of classification compare to the previous (10-epoch) model? Is it just less "noisy", or is there something else going on?

## Occlusion
What is the model looking at? Let's try out our occlusion probing to see what we can learn about what it considered important.

In [ ]:

def test_image(img, filename, labels):
    img_array = tf.keras.utils.img_to_array(img)
    img_array = preprocess_input_MWLR(img_array) # Rescale the image
    img_array = tf.expand_dims(img_array, 0)  # Create a batch
    predictions = model.predict(img_array, verbose=0)
    scores = predictions[0]
    score = np.max(scores)
    ans = labels[np.argmax(scores)]
    print(f"{filename}: {ans} {score}")
    return score


def occlude(image_file, labels_path, occlude_size=1, fill=0.5, actual_class = False):

    with open(labels_path, 'r') as f:
        labels = [l for l in f.read().split('\n')]    

    # Test the main image first
    img = tf.keras.utils.load_img(os.path.join(test_data_dir, image_file), target_size=IMAGE_SIZE_XCEPTION) # PIL format
    
    figure, axis = plt.subplots(1, 2)
    axis[0].set_axis_off()
    #axis[0].imshow(img)
    
    base_score = test_image(img, image_file, labels)

    fill = int(fill*255)
    scores = []
    mask_size = int(IMAGE_SIZE_XCEPTION[0]/occlude_size)
   
    for x in range(mask_size):
        for y in range(mask_size):
            img = tf.keras.utils.load_img(os.path.join(test_data_dir, image_file), target_size=IMAGE_SIZE_XCEPTION) # PIL format
            draw = ImageDraw.Draw(img)
            xx = x*occlude_size
            yy = y*occlude_size
            draw.rectangle([xx, yy, xx + occlude_size - 1, yy + occlude_size - 1], fill=(fill,fill,fill)) # Grey
            if x==1 and y==1:
                axis[0].imshow(img)
            score = test_image(img, image_file, labels)
            # print(f"({y},{x}): {base_score - score}")
            scores.append(base_score - score)
    
    scores -= np.min(scores)
    scores /= np.max(scores)
    print("Scores:")
    print(scores)
    # scores = scores.reshape(mask_size, mask_size)
    
    # Display it
    mask = Image.new("RGBA", (mask_size, mask_size), (0,0,0))
    draw = ImageDraw.Draw(mask)
    for x in range(mask_size):
        for y in range(mask_size):
            # score = int(scores[x, y] * 255)
            score = int(scores[x*mask_size + y] * 255)
            draw.rectangle([x, y, x+1, y+1], fill=(score,score,score))
            
    axis[1].set_axis_off()
    axis[1].imshow(mask)

test_dir = 'files/moths_data/test'
# occlude_image = 'Cebysa_leucotelus/Cebysa leucotelus_m1a.png'
occlude_image = 'Tyria_jacobaeae/Tyria jacobaea1a.png'
# occlude_image = 'Hyphantria_cunea/Hyphantria cunea3_red race_m_a.png'
occlude(occlude_image, labels_path, occlude_size=20, fill=0.5, actual_class=False)

## Summary
- We can train models for our specific domains, even from a relatively small number of images, using transfer learning
- A confusion matrix is useful for examining mis-classification patterns
- Occlusion testing can confirm that the model has learned the relevant features, or expose where spurious differences in the imagery have been modelled, such as a change of background, lighting conditions or camera characteristics.

[Next: segmentation](/notebooks/5_segmentation.ipynb)